In [ ]:
%cd ..

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import glob

from src.model.yolo_handler import YoloHandler
from src.model.spn_handler import SPNHandler
from utils.config_parser import ConfigParser
from pruning.channel_selection.channel_selector import ChannelSelector
from state_predictor.coder import Coder
from utils.common_utils import normalize, denormalize

# Init report
report_df = pd.DataFrame(columns=["original_params", "original_mAP",
                                   "pruned_params", "pruned_mAP", "pruned_mAP_drop", "pruned_spars", "pruned_dmap", "pruned_spars/dmap",
                                   "finetune_epochs", "finetune_params", "finetune_mAP", "finetune_mAP_drop", "finetune_spars", "finetune_dmap", "finetune_spars/dmap",
                                   "test_case", "run_name", "episode", "alpha_sequence"])

## Pruning Functions


In [ ]:
def eval_pruned(alpha_sequence, model_handler, channel_selector, do_print = False):

    # Collect initial model layer info
    init_model_channels = [layer[1].out_channels for layer in model_handler.prunable_layers]

    # Select indices and prune model layer-wise
    all_indices = [None] * model_handler.n_prunable_layers
    for i, layer in enumerate(model_handler.prunable_layers):
        idxs = channel_selector.select_indices(layer[1], alpha_sequence[i])
        all_indices[i] = idxs
        model_handler.prune(all_indices, i)
        model_handler.determine_prunable_layers()

    # Collect pruned model layer info
    pruned_model_channels = [layer[1].out_channels for layer in model_handler.prunable_layers]

    # Print statistics
    if do_print:
        for i, layer in enumerate(model_handler.prunable_layers):
            n_removed_channels = init_model_channels[i] - pruned_model_channels[i]
            print(f"Layer {i:<3} {layer[0]:<20} a = {alpha_sequence[i]:<5} {n_removed_channels:<3}/{init_model_channels[i]:<5} channels removed.")

    metrics = model_handler.evaluate()
    return metrics


def eval_spn(alpha_sequence: np.ndarray, yolo_handler, spn_handler, coder, device, state_features, alpha_range, do_print_diagnostics=False):

    batch_size = 1
    predictions = []

    # Init environment
    action_batch = torch.full([batch_size, 1, yolo_handler.n_prunable_layers], -1.0).to(device)
    state_batch = torch.full([batch_size, len(state_features)-1, yolo_handler.n_prunable_layers], 1.0).to(device)
    
    for layer_i, _ in enumerate(yolo_handler.prunable_layers):

        action = alpha_sequence[layer_i]   
        action_batch[0, :, layer_i] = normalize(action, value_range=alpha_range)             
       
        spn_input_data = torch.cat((action_batch, state_batch), dim=1).view([batch_size, -1])   #.permute(0,2,1).flatten().unsqueeze(0)
        prediction = spn_handler.predict(spn_input_data)
        sparsb, dmapb = prediction[0], prediction[1]
        decoded_prediction = coder.decode_label(prediction) # Tuple([batch_size], [batch_size])

        # Update state batc        
        with torch.no_grad():
            state_batch = state_batch.clone()
            #state_batch[0, 0, layer_i] = 1
            state_batch[0, 0, layer_i] = sparsb
            state_batch[0, 1, layer_i] = dmapb

        predictions.append(decoded_prediction)

        if layer_i == 92 and do_print_diagnostics:
            df = pd.DataFrame(state_batch[0].T.cpu().numpy(), columns=['spars', 'dmap']).round(3)
            for i, row in df.iterrows():
                print(f"Layer {i:>2}: spars = {row['spars']:.3f}, dmap = {row['dmap']:.3f}")

    
    return predictions, state_batch, action_batch


def eval_transformer_spn(alpha_sequence: np.ndarray, spn_handler, alpha_range, do_print_diagnostics=False):

    norm_alpha_sequence = normalize(alpha_sequence, value_range=alpha_range)         
    print(norm_alpha_sequence)    
    predictions = spn_handler.autoregressive_predict(norm_alpha_sequence)

    return predictions


## MAIN

### Define alpha sequence

#### Define Manually

In [ ]:
norm_alpha =    [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 0.0, -1.0, -1.0, -1.0, -1.0]  

init_alpha_sequence = denormalize(np.array(norm_alpha), value_range=(0, 2.0))

# Log for the Report
report_df["alpha_sequence"] = [init_alpha_sequence.tolist()]
report_df["test_case"] = ["manual"]

#### Load from the DB

In [ ]:

# Load from saved sample
#sample_data_path = "/data/blanka/DATASETS/SPN/YOLOv8x_7030_modelwise_shifted_zerofilled_extra/validation/data"
sample_data_path = "/data/blanka/DATASETS/SPN/YOLOv8x/data"
sample_pattern = "181_*.pkl"
# Find matching data file
data_files = glob.glob(os.path.join(sample_data_path, sample_pattern))
if not data_files:
    raise FileNotFoundError(f"No data files matching pattern {sample_pattern}")
data_file = data_files[0]
state_df = pd.read_pickle(data_file)
init_alpha_sequence =  state_df["alpha"].to_numpy()[:93]

alpha_sequence = init_alpha_sequence
print(len(alpha_sequence))
print(state_df)

# Log for the Report
report_df["alpha_sequence"] = [alpha_sequence.tolist()]
report_df["test_case"] = ["DB"]
report_df["run_name"] = [sample_data_path]
report_df["episode"] = [sample_pattern]

#### Load from the Results

In [ ]:
test_case = "modelwise_extra_transf_stateshifttest_PTSreward_3actions_accurew"
run_name = "20251106_152210_5e65f9_20251101_054201_3ee1b7_optuna_reprod"
episode = 45
alpha_value_range = (0.0, 2.0)
base_path = "/data/blanka/runs/RL/YOLOv8x"

path = os.path.join(base_path, test_case, run_name, "logs", "bests.pkl")
results_df = pd.read_pickle(path)

reward = results_df.iloc[episode]["reward"]
spars_norm =  results_df.iloc[episode]["spars"]
dmap_norm = results_df.iloc[episode]["dmap"]
alpha_sequence_norm = results_df.iloc[episode]["alpha_seq"]

spars = denormalize(spars_norm, value_range=(0.0, 1.0))
dmap = denormalize(dmap_norm, value_range=(0.0, 1.0))   
alpha_sequence = denormalize(np.array(alpha_sequence_norm), value_range=alpha_value_range)
init_alpha_sequence = alpha_sequence

print(f"Episode {episode}: reward = {reward}, spars_norm = {spars_norm}, dmap_norm = {dmap_norm}")
print(f"Episode {episode}: reward = {reward}, spars = {spars}, dmap = {dmap}")
print(f"Norm alpha sequence: {alpha_sequence_norm}")
print(f"Init alpha sequence: {alpha_sequence}")

# Log for the Report
report_df["alpha_sequence"] = [alpha_sequence.tolist()]
report_df["test_case"] = [test_case]
report_df["run_name"] = [run_name]
report_df["episode"] = [episode]

### Load Handlers & Configs

In [ ]:
spn_run_name = "20250809_230714_dee9e0_optuna"
spn_conf = ConfigParser.read(f"/data/blanka/runs/SPN/YOLOv8x/prevfeat_modelwise_shifted_zerofilled_extra_transf/{spn_run_name}/settings.ini")
pruning_conf = ConfigParser.read("config/pruning/pruning_sampling.ini")

yolo_handler = YoloHandler(pruning_conf.model)
spn_handler = SPNHandler(spn_conf, run_name=spn_run_name, tb_handler=None)
spn_handler.create(is_pretrained=True)

channel_selector = ChannelSelector(pruning_conf.channel_selection)

alpha_range = pruning_conf.alpha.min_max_steps[:2]
state_features = spn_conf.model.state_features
device = spn_conf.train.device


# filtered_state_df = state_df[conf.model.state_features]        
# encoded_state = coder.encode_state(filtered_state_df, label_df)
# encoded_label = coder.encode_label(label_df)

# # load or define SPN model
# pred_spars, pred_dmap = spn_handler.predict(encoded_state.unsqueeze(dim=0))


In [ ]:
yolo_handler.determine_prunable_layers()
print(yolo_handler.n_prunable_layers)

### Evaluate Init Model & Define Coder

In [ ]:
# Determine prunable layers
yolo_handler.determine_prunable_layers()
assert len(alpha_sequence) == yolo_handler.n_prunable_layers, (
f"Alpha_sequence length ({len(alpha_sequence)}) is not equal to the number of prunable layers ({yolo_handler.n_prunable_layers})!"
)

# Evaluate initial model
init_metrics = yolo_handler.evaluate()

# Log init metrics for the Report
report_df["original_params"] = [init_metrics[4]]
report_df["original_mAP"] = [init_metrics[2]*100]

# Load example 
sample_data_path_ex = "/data/blanka/DATASETS/SPN/YOLOv8x/data"
sample_pattern_ex = "0_*.pkl"
# Find matching data file
data_files_ex = glob.glob(os.path.join(sample_data_path_ex, sample_pattern_ex))
if not data_files_ex:
    raise FileNotFoundError(f"No data files matching pattern {sample_pattern_ex}")
data_file_ex = data_files_ex[0]
example_state_df = pd.read_pickle(data_file_ex)


coder = Coder(state_example=example_state_df, label_example=None, alpha_range=alpha_range)

### Prune & evaluate

In [ ]:
# Eval at layer
layer_to_be_evaluated = 92
alpha_sequence = np.where(np.arange(len(init_alpha_sequence)) <= layer_to_be_evaluated, init_alpha_sequence, 0)
print(alpha_sequence)

pruned_metrics = eval_pruned(alpha_sequence, yolo_handler, channel_selector)


In [ ]:
# Construct label
metric_features = ['recall', 'precision', 'map50', 'map90', 'n_params']
init_columns = [col + '_init' for col in metric_features]
label_df = pd.DataFrame([],columns=metric_features + ["n_layer_channels"] + init_columns)

label_df.loc[0, metric_features] = pruned_metrics
label_df.loc[0, 'n_layer_channels'] = 0
label_df.loc[0, init_columns] = init_metrics

encoded_metrics = coder.encode_label(label_df)
decoded_true_spars = denormalize(encoded_metrics[0], value_range=(0, 1))
decoded_true_dmap = denormalize(encoded_metrics[1], value_range=(0, 1))

print(f"Layer {layer_to_be_evaluated}\t encode_spars: {encoded_metrics[0]:.3f}\t encode_dmap: {encoded_metrics[1]:.3f}")
print(f"Layer {layer_to_be_evaluated}\t spars: {decoded_true_spars:.3f}\t dmap: {decoded_true_dmap:.3f}")

# Logging for the Report
report_df["pruned_params"] = [pruned_metrics[4]]
report_df["pruned_mAP"] = [pruned_metrics[2]*100]
report_df["pruned_spars"] = [round(decoded_true_spars.item()*100, 2)]
report_df["pruned_dmap"] = [round(decoded_true_dmap.item()*100, 2)]

In [ ]:
# Save pruned and load after

#yolo_handler.save_pruned_model()
yolo_handler_pruned = YoloHandler(pruning_conf.model)
yolo_handler_pruned.load_pruned_model(path="runs/pruned/pruned_model.pt")
yolo_handler_pruned.evaluate()

### Fine-tuning

In [ ]:
ft_epocshs = 1
yolo_handler.fine_tune(data_yaml="/home/blanka/Multi-Domain-Pruning/config/data/kitti.yaml", epochs=ft_epocshs)

In [ ]:
ft_metrics = yolo_handler.evaluate()

# Construct label
metric_features = ['recall', 'precision', 'map50', 'map90', 'n_params']
init_columns = [col + '_init' for col in metric_features]
ft_label_df = pd.DataFrame([],columns=metric_features + ["n_layer_channels"] + init_columns)

ft_label_df.loc[0, metric_features] = ft_metrics
ft_label_df.loc[0, 'n_layer_channels'] = 0
ft_label_df.loc[0, init_columns] = init_metrics

encoded_ft_metrics = coder.encode_label(ft_label_df)
decoded_ft_true_spars = denormalize(encoded_ft_metrics[0], value_range=(0, 1))
decoded_ft_true_dmap = denormalize(encoded_ft_metrics[1], value_range=(0, 1))

print(f"Layer {layer_to_be_evaluated}\t finetune encode_spars: {encoded_ft_metrics[0]:.3f}\t finetune encode_dmap: {encoded_ft_metrics[1]:.3f}")
print(f"Layer {layer_to_be_evaluated}\t finetune spars: {decoded_ft_true_spars:.3f}\t finetune dmap: {decoded_ft_true_dmap:.3f}")

print(ft_metrics)

# Logging for the Report
report_df["finetune_epochs"] = [ft_epocshs]
report_df["finetune_params"] = [ft_metrics[4]]
report_df["finetune_mAP"] = [ft_metrics[2]*100]
report_df["finetune_spars"] = [round(decoded_ft_true_spars.item()*100, 2)]
report_df["finetune_dmap"] = [round(decoded_ft_true_dmap.item()*100, 2)]

In [ ]:
yolo_handler.save_pruned_model(path="runs/pruned/fine_tuned_pruned_model.pt")
yolo_handler_pruned = YoloHandler(pruning_conf.model)
yolo_handler_pruned.load_pruned_model(path="runs/pruned/fine_tuned_pruned_model.pt")
yolo_handler_pruned.evaluate()

### Get the sample from the DB (IF RELEVANT)

In [ ]:
filtered_state_df = state_df[state_features]   
encoded_state = coder.encode_state(filtered_state_df, label_df)
encoded_state = encoded_state.reshape([3, -1]).T
print(encoded_state)
pred_spars, pred_dmap = spn_handler.predict(encoded_state)

df = pd.DataFrame(encoded_state, columns=['alpha', 'spars', 'dmap']).round(3)
#df = pd.DataFrame(encoded_state.reshape(93,3), columns=['alpha', 'spars', 'dmap']).round(3)
spn_preds_from_gt = df
# for i, row in df.iterrows():
#     print(f"Layer {i:>2}: spars = {row['spars']:.3f}, dmap = {row['dmap']:.3f}")

# Denormalize the encoded label
decoded_pred_spars = denormalize(pred_spars, value_range=(0, 1))
decoded_pred_dmap = denormalize(pred_dmap, value_range=(0, 1))
print(f"Layer 92 \t spars: {decoded_pred_spars.item():.3f}\t dmap: {decoded_pred_dmap.item():.3f}")

### Predict with SPN

#### MLP SPN

In [ ]:
spn_predictions_decoded, batch_state, actions_to_plot = eval_spn(alpha_sequence, yolo_handler, spn_handler, coder, device, state_features, alpha_range, do_print_diagnostics=False)
spn_preds_from_scratch = batch_state

# print("\n\n ####### Results Denormalized #######")
# for i, (pred, alpha) in enumerate(zip(spn_predictions_decoded, alpha_sequence)):

#     tab = "\t" if i>9 else "\t\t"
#     print(f"Layer {i}{tab} {alpha}\t spars: {pred['spars'].item():.3f}\t dmap: {pred['dmap'].item():.3f}")


#### Transformer SPN

In [ ]:
spn_preds_from_scratch = eval_transformer_spn(alpha_sequence, spn_handler, alpha_range, do_print_diagnostics=False)

print(spn_preds_from_scratch)


## Generate REPORT

In [ ]:
# Calculate missing metrics for the Report

def is_filled(*cols):
    return all(c in report_df and report_df[c].notna().any() for c in cols)

if is_filled("original_mAP", "pruned_mAP"):
    report_df["pruned_mAP_drop"] = report_df["original_mAP"] - report_df["pruned_mAP"]

if is_filled("pruned_spars", "pruned_dmap"):
    report_df["pruned_spars/dmap"] = round(report_df["pruned_spars"] / report_df["pruned_dmap"], 2)

if is_filled("original_mAP", "finetune_mAP"):
    report_df["finetune_mAP_drop"] = report_df["original_mAP"] - report_df["finetune_mAP"]

if is_filled("finetune_spars", "finetune_dmap"):
    report_df["finetune_spars/dmap"] = round(report_df["finetune_spars"] / report_df["finetune_dmap"], 2)


# Print 
    
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Don't wrap lines
pd.set_option('display.max_colwidth', None)  # Show full contents of each cell

print(report_df)


### Save the Report

In [ ]:

# Save Report
base_path = "/data/blanka/REPORTS/YOLOv8x"
report_path = os.path.join(base_path, f"{test_case}___{run_name}___ep{episode}.csv")
report_df.to_csv(report_path, index=False)

## PLOT PREDICTIONS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

# Suppose your data is already loaded
data_file_spars = 'spn_preds_spars.csv'
data_file_dmap = 'spn_preds_dmap.csv'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot for "spars"
x_spars = np.arange(len(spn_preds_from_gt['spars']))
y1_spars = spn_preds_from_gt['spars'].reset_index(drop=True)
y2_spars = spn_preds_from_scratch[:, 0].cpu()
#alpha = norm_alpha

axes[0].plot(x_spars, y1_spars, label='from_gt')
axes[0].plot(x_spars, y2_spars, label='from_scratch')
#axes[0].plot(x_spars, alpha, label='alpha')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Spars Value')
axes[0].set_title(data_file_spars)
axes[0].legend()
axes[0].grid()



# Plot for "dmap"
x_dmap = np.arange(len(spn_preds_from_gt['dmap']))
y1_dmap = spn_preds_from_gt['dmap'].reset_index(drop=True)
y2_dmap = spn_preds_from_scratch[:, 1].cpu()

axes[1].plot(x_dmap, y1_dmap, label='from_gt')
axes[1].plot(x_dmap, y2_dmap, label='from_scratch')
#axes[1].plot(x_dmap, alpha, label='alpha')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('Dmap Value')
axes[1].set_title(data_file_dmap)
axes[1].legend()
axes[1].grid()


plt.tight_layout()
plt.show()
